In [2]:
import openmeteo_requests

from openmeteo_sdk.Variable import Variable
from openmeteo_sdk.Aggregation import Aggregation

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


In [3]:
import pandas as pd
import numpy as np

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://ensemble-api.open-meteo.com/v1/ensemble"
params = {
	"latitude": 40.0951,
	"longitude": -75.6169,
    "timezone": "America/New_York",
	"temperature_unit": "fahrenheit",
	"wind_speed_unit": "mph",
	"hourly": ["temperature_2m", "precipitation", "wind_speed_10m"],
	"current": ["temperature_2m", "relative_humidity_2m"],
	"models": "google_weathernext2_ensemble",
    "start_date": "2026-07-18",
	"end_date": "2026-07-18",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_variables = list(map(lambda i: hourly.Variables(i), range(0, hourly.VariablesLength())))
hourly_temperature_2m = filter(lambda x: x.Variable() == Variable.temperature and x.Altitude() == 2, hourly_variables)
hourly_wind_speed_10m = filter(lambda x: x.Variable() == Variable.wind_speed and x.Altitude() == 2, hourly_variables)
hourly_precipitation = filter(lambda x: x.Variable() == Variable.precipitation and x.Altitude() == 2, hourly_variables)


# extract numpy arrays (or empty arrays)
# consume the filter iterators and replace them with the first matching VariableWithValues (or None)
temp_var = next(hourly_temperature_2m, None)
hourly_temperature_2m = temp_var
precip_var = next(hourly_precipitation, None)
hourly_precipitation = precip_var
wind_var = next(hourly_wind_speed_10m, None)
hourly_wind_speed_10m = wind_var

temp_vals = temp_var.ValuesAsNumpy() if temp_var is not None else np.array([])
precip_vals = hourly_precipitation.ValuesAsNumpy() if hourly_precipitation is not None else np.array([])
wind_vals = hourly_wind_speed_10m.ValuesAsNumpy() if hourly_wind_speed_10m is not None else np.array([])

# build date index with length matching available data (prefer temperature length if present)
if temp_vals.size > 0:
    n = temp_vals.shape[0]
elif precip_vals.size > 0:
    n = precip_vals.shape[0]
elif wind_vals.size > 0:
    n = wind_vals.shape[0]
else:
    # fallback to original time range
    hourly_data = {"date": pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
        end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    )}
    n = len(hourly_data["date"])

date_index = pd.date_range(
    start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
    periods = n,
    freq = pd.Timedelta(seconds = hourly.Interval()),
    inclusive = "left"
)

hourly_data = {"date": date_index}
if temp_vals.size > 0:
    hourly_data["temperature_2m"] = temp_vals
if precip_vals.size > 0:
    hourly_data["precipitation"] = precip_vals
if wind_vals.size > 0:
    hourly_data["wind_speed_10m"] = wind_vals

hourly_dataframe_pd = pd.DataFrame(data = hourly_data)
print(hourly_dataframe_pd)

Coordinates: 40.0°N -75.5°E
Elevation: 81.0 m asl
Timezone difference to GMT+0: -14400s
                        date  temperature_2m
0  2026-07-18 04:00:00+00:00       74.995697
1  2026-07-18 05:00:00+00:00       73.645699
2  2026-07-18 06:00:00+00:00       72.655701
3  2026-07-18 07:00:00+00:00       71.845703
4  2026-07-18 08:00:00+00:00       71.215698
5  2026-07-18 09:00:00+00:00       70.765701
6  2026-07-18 10:00:00+00:00       70.495697
7  2026-07-18 11:00:00+00:00       70.495697
8  2026-07-18 12:00:00+00:00       70.675697
9  2026-07-18 13:00:00+00:00       71.305702
10 2026-07-18 14:00:00+00:00       72.295700
11 2026-07-18 15:00:00+00:00       73.555702
12 2026-07-18 16:00:00+00:00       74.815697
13 2026-07-18 17:00:00+00:00       75.985695
14 2026-07-18 18:00:00+00:00       76.885696
15 2026-07-18 19:00:00+00:00       77.515701
16 2026-07-18 20:00:00+00:00       77.965698
17 2026-07-18 21:00:00+00:00       78.325699
18 2026-07-18 22:00:00+00:00       78.505699
19 2026-07-1

In [ ]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://ensemble-api.open-meteo.com/v1/ensemble"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"hourly": "temperature_2m",
	"models": "google_weathernext2_ensemble",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_variables = list(map(lambda i: hourly.Variables(i), range(0, hourly.VariablesLength())))
hourly_temperature_2m = filter(lambda x: x.Variable() == Variable.temperature and x.Altitude() == 2, hourly_variables)

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

# Process all hourly members
for variable in hourly_temperature_2m:
	member = variable.EnsembleMember()
	hourly_data[f"temperature_2m_member{member}"] = variable.ValuesAsNumpy()

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)